In [1]:
import pandas as pd

In [2]:
!wget https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb
%run db2.ipynb
db2creds_file = 'db2con.env'
from dotenv import dotenv_values
db2creds = dotenv_values(db2creds_file)

--2024-06-03 18:34:54--  https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 155540 (152K) [text/plain]
Saving to: ‘db2.ipynb.10’

db2.ipynb.10        100%[===================>] 151.89K  --.-KB/s    in 0.01s   

2024-06-03 18:34:54 (11.5 MB/s) - ‘db2.ipynb.10’ saved [155540/155540]



<>:1692: SyntaxWarning: invalid escape sequence '\s'
<>:2285: SyntaxWarning: invalid escape sequence '\?'
/tmp/ipykernel_18683/1557473136.py:1692: SyntaxWarning: invalid escape sequence '\s'
  firstCommand = "(?:^\s*)([a-zA-Z]+)(?:\s+.*|$)"
/tmp/ipykernel_18683/1557473136.py:2285: SyntaxWarning: invalid escape sequence '\?'
  pattern = "\?\*[0-9]+"


Db2 Extensions Loaded. Version: 2024-05-29


In [3]:
%sql CONNECT CREDENTIALS db2creds

Connection successful. tpcds @ localhost 


In [4]:
df_queries_columns = ['query_id', 'appl_id', 'uow_id', 'activity_id', 'explain_time', 'query']

In [5]:
df_queries = pd.read_csv('success.csv', header=None, names=df_queries_columns)

In [6]:
df_queries.shape

(5764, 6)

In [7]:
df_queries.head(5)

,query_id,appl_id,uow_id,activity_id,explain_time,query
0,1,*LOCAL.shaikhq.240530151621,4,1,2024-05-30-08.16.11.238384,"SELECT TPCDS.CUSTOMER.C_BIRTH_YEAR , TPCDS.DAT..."
1,2,*LOCAL.shaikhq.240530151621,12,1,2024-05-30-08.16.12.871381,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
2,4,*LOCAL.shaikhq.240530151621,24,1,2024-05-30-08.16.24.574183,"SELECT TPCDS.WEB_SITE.WEB_CLOSE_DATE_SK , TPCD..."
3,5,*LOCAL.shaikhq.240530151621,32,1,2024-05-30-08.16.25.771665,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
4,6,*LOCAL.shaikhq.240530151621,40,1,2024-05-30-08.16.26.997895,"SELECT TPCDS.CATALOG_PAGE.CP_CATALOG_PAGE_SK ,..."


In [21]:
query1_ts = "2024-05-30-08.16.11.238384"
df_queries = df_queries[df_queries['explain_time'] == query1_ts]
activity_id = df_queries['activity_id'].values[0]
appl_id = df_queries['appl_id'].values[0]
uow_id = df_queries['uow_id'].values[0]

# collecting query level stats

In [22]:
# collecting final actual card
sql = f""" 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

In [23]:
print(sql)

 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = 1 AND 
APPL_ID = '*LOCAL.shaikhq.240530151621' AND 
UOW_ID = 4



In [24]:

df_activity = %sql {sql}

In [25]:

actual_card = df_activity['ROWS_RETURNED'].values[0]
sort_shrheap_top = df_activity['SORT_SHRHEAP_TOP'].values[0]

In [26]:
sql = f""" 
SELECT STMT_EXEC_TIME 
FROM ACTIVITYMETRICS_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

df_activitymetrics = %sql {sql}
stmt_exec_time = df_activitymetrics['STMT_EXEC_TIME'].values[0]

In [27]:
print('actual_card: {}'.format(actual_card))
print('sort_shrheap_top: {}'.format(sort_shrheap_top))
print('stmt_exec_time: {}'.format(stmt_exec_time))

actual_card: 8159
sort_shrheap_top: 69
stmt_exec_time: 365


# Collecting Node level information for each node

In [ ]:
# find out the the nodes / operators
# fetching operators
sql = f"""
SELECT OPERATOR_ID, OPERATOR_TYPE 
FROM EXPLAIN_OPERATOR
WHERE EXPLAIN_TIME = '{explain_time}'
"""

df_explain_operator = %sql {sql}
print(df_explain_operator)
operator_ids = df_explain_operator['OPERATOR_ID'].tolist()
print(operator_ids)

   OPERATOR_ID OPERATOR_TYPE
0            1        RETURN
1            2        HSJOIN
2            3        TBSCAN
3            4        TBSCAN
[1, 2, 3, 4]


In [ ]:
df_explain_operator[df_explain_operator['OPERATOR_ID'] == 4]['OPERATOR_TYPE'].values[0]

'TBSCAN'

In [ ]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')
stream_cols = ['SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'COLUMN_NAMES']
df_explain_stream = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == explain_time)][stream_cols]

In [ ]:
df_explain_stream.head()

,SOURCE_ID,TARGET_TYPE,TARGET_ID,OBJECT_NAME,STREAM_COUNT,COLUMN_COUNT,COLUMN_NAMES
0,-1,O,3,CUSTOMER,100000.000000,3,+Q1.$RID$+Q1.C_BIRTH_YEAR+Q1.C_FIRST_SHIPTO_DA...
1,3,O,2,NaN,100000.000000,-1,NaN
2,-1,O,4,DATE_DIM2,73049.000000,5,+Q2.$RID$+Q2.D_WEEK_SEQ+Q2.D_YEAR+Q2.D_MOY+Q2....
3,4,O,2,NaN,3983.533936,-1,NaN
4,2,O,1,NaN,104386.367188,-1,NaN


In [ ]:
df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')

In [ ]:
df_explain_predicate['HOW_APPLIED'].unique()

array(['JOIN      ', 'SARG      ', 'START     ', 'STOP      ',
       'GAP_START ', 'GAP_STOP  ', 'DPSTART   ', 'DPSTOP    ',
       'RESID     '], dtype=object)

In [ ]:
df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')
predicate_cols = [ 'OPERATOR_ID',
       'PREDICATE_ID', 'HOW_APPLIED', 'WHEN_EVALUATED', 'RELOP_TYPE',
       'SUBQUERY', 'FILTER_FACTOR', 'PREDICATE_TEXT']

df_predicate_filtered = df_explain_predicate[df_explain_predicate['EXPLAIN_TIME'] == explain_time][predicate_cols]
print('df_predicate_filtered shape :{}'.format(df_predicate_filtered.shape))
print('df_predicate_filtered: ', df_predicate_filtered)

df_predicate_filtered shape :(3, 8)
df_predicate_filtered:     OPERATOR_ID  PREDICATE_ID HOW_APPLIED WHEN_EVALUATED RELOP_TYPE SUBQUERY  \
0            2             2  JOIN                              EQ        N   
1            4             3  SARG                              LE        N   
2            4             4  SARG                              EQ        N   

   FILTER_FACTOR                              PREDICATE_TEXT  
0       0.000014  (Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)  
1       0.713558                         (1958 <= Q2.D_YEAR)  
2       0.081000                             (Q2.D_MOY = 12)  


In [ ]:
df_predicate_filtered

,OPERATOR_ID,PREDICATE_ID,HOW_APPLIED,WHEN_EVALUATED,RELOP_TYPE,SUBQUERY,FILTER_FACTOR,PREDICATE_TEXT
0,2,2,JOIN,,EQ,N,0.000014,(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)
1,4,3,SARG,,LE,N,0.713558,(1958 <= Q2.D_YEAR)
2,4,4,SARG,,EQ,N,0.081000,(Q2.D_MOY = 12)


In [ ]:
op_id = 3
df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == op_id]['PREDICATE_TEXT'].values

array([], dtype=object)

In [ ]:
df_explain_stream.shape

(5, 7)

In [ ]:
print(df_explain_stream['COLUMN_NAMES'][0])

+Q1.$RID$+Q1.C_BIRTH_YEAR+Q1.C_FIRST_SHIPTO_DATE_SK


In [ ]:
nodes = {}

for operator_id in operator_ids:
    node_dict = {}
    node_dict['Node Type'] = df_explain_operator[df_explain_operator['OPERATOR_ID'] == operator_id]['OPERATOR_TYPE'].values[0]
    
    # check if there is any relation / table involved in this operation
    relation_name = df_explain_stream[(df_explain_stream['TARGET_ID'] == operator_id) 
                                      & (df_explain_stream['OBJECT_NAME'].notna())]['OBJECT_NAME'].values
    
    if len(relation_name) > 0:
        node_dict['Relation Name'] = relation_name[0]
        # collecting local predicates, if any
        local_predicate = df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == operator_id]['PREDICATE_TEXT'].values
        if len(local_predicate) > 0:
            node_dict['Local Predicate'] = local_predicate.tolist()
        
    # check if there is any join predicate
    join_predicate = df_predicate_filtered[(df_predicate_filtered['OPERATOR_ID'] == operator_id) & 
                          (df_predicate_filtered['HOW_APPLIED'].str.strip() == 'JOIN')]['PREDICATE_TEXT'].values
    
    if len(join_predicate) > 0:
        node_dict['Join Predicate'] = join_predicate[0]
    
    nodes[operator_id] = node_dict


In [ ]:
nodes

{1: {'Node Type': 'RETURN'},
 2: {'Node Type': 'HSJOIN',
  'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)'},
 3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Local Predicate': ['(1958 <= Q2.D_YEAR)', '(Q2.D_MOY = 12)']}}

In [ ]:
df_explain_stream

,SOURCE_ID,TARGET_TYPE,TARGET_ID,OBJECT_NAME,STREAM_COUNT,COLUMN_COUNT,COLUMN_NAMES
0,-1,O,3,CUSTOMER,100000.000000,3,+Q1.$RID$+Q1.C_BIRTH_YEAR+Q1.C_FIRST_SHIPTO_DA...
1,3,O,2,NaN,100000.000000,-1,NaN
2,-1,O,4,DATE_DIM2,73049.000000,5,+Q2.$RID$+Q2.D_WEEK_SEQ+Q2.D_YEAR+Q2.D_MOY+Q2....
3,4,O,2,NaN,3983.533936,-1,NaN
4,2,O,1,NaN,104386.367188,-1,NaN


In [283]:

# Identifying parent-child relationship
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[stream_cols]

KeyError: 'EXPLAIN_TIME'

In [ ]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN'}
2: {'Node Type': 'HSJOIN', 'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)'}
3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'}
4: {'Node Type': 'TBSCAN', 'Relation Name': 'DATE_DIM2', 'Local Predicate': ['(1958 <= Q2.D_YEAR)', '(Q2.D_MOY = 12)']}


# Scratchpad section

In [ ]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')

In [ ]:
df_explain_stream.columns

Index(['EXPLAIN_REQUESTER', 'EXPLAIN_TIME', 'SOURCE_NAME', 'SOURCE_SCHEMA',
       'SOURCE_VERSION', 'EXPLAIN_LEVEL', 'STMTNO', 'SECTNO', 'STREAM_ID',
       'SOURCE_TYPE', 'SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_SCHEMA',
       'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'PREDICATE_ID',
       'COLUMN_NAMES', 'PMID', 'SINGLE_NODE', 'PARTITION_COLUMNS',
       'SEQUENCE_SIZES', 'OBJECT_TENANTID'],
      dtype='object')

In [ ]:
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == explain_time)][stream_cols]

In [257]:
nodes

{1: {'Node Type': 'RETURN'},
 2: [4],
 3: {'Node Type': 'TBSCAN',
  'Relation Name': 'CUSTOMER',
  'parent': 2,
  'Parent': 2},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Local Predicate': ['(1958 <= Q2.D_YEAR)', '(Q2.D_MOY = 12)'],
  'parent': 2,
  'Parent': 2}}

In [255]:
for index, row in df_stream_filtered.iterrows():
    source_id = row['SOURCE_ID']
    target_id = row['TARGET_ID']
    if source_id > 0:
        nodes[source_id]['Parent'] = target_id
        # if 'Children' not in nodes[target_id]:
        #     nodes[target_id] = []
        # nodes[target_id].append(source_id)
    # print('source: {}, target: {}'.format(source_id, target_id))

TypeError: list indices must be integers or slices, not str

In [251]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN'}
2: {'Node Type': 'HSJOIN', 'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)', 'parent': 1}
3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER', 'parent': 2}
4: {'Node Type': 'TBSCAN', 'Relation Name': 'DATE_DIM2', 'Local Predicate': ['(1958 <= Q2.D_YEAR)', '(Q2.D_MOY = 12)'], 'parent': 2}


In [253]:
type(nodes)

dict

In [254]:
type(nodes[1])

dict